# Face Recognition Classifier Evaluation
### Class Monitoring System Using Computer Vision for Automated Attendance Management

This notebook loads the trained SVM and KNN classifiers (trained on 128-d
`face_recognition` embeddings) and produces the full evaluation report used
in the thesis Results chapter:

1. Train/test split summary
2. Per-model metrics (accuracy, precision, recall, F1)
3. Confusion matrices (visualised)
4. Side-by-side SVM vs KNN comparison table and chart

**Important limitation to note here and in the thesis:** with a small
number of registered students and only 2-3 images each, the test set per
class is very small. Metrics below should be read as a *methodology
demonstration*, not as a statistically robust accuracy claim. This is
flagged again at the bottom of the notebook.

> Run `encode_faces.py` then `train_classifier.py` from the project root
> *before* running this notebook — it loads their saved outputs.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# Project paths (notebook lives in ml_pipeline/, project root is one level up)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ml_pipeline" else Path.cwd()
MODELS_DIR = PROJECT_ROOT / "ml_pipeline" / "models"
ENCODINGS_PATH = PROJECT_ROOT / "data" / "encodings" / "encodings.pkl"

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

## 1. Load trained artifacts

In [ ]:
with open(MODELS_DIR / "evaluation_results.pkl", "rb") as f:
    results = pickle.load(f)

with open(MODELS_DIR / "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

with open(ENCODINGS_PATH, "rb") as f:
    encoding_data = pickle.load(f)

svm_results = results["svm"]
knn_results = results["knn"]
class_names = label_encoder.classes_

print(f"Registered students ({len(class_names)}): {list(class_names)}")
print(f"Total encoded images used for train/test: {len(encoding_data['raw_labels'])}")

## 2. Dataset composition (images per student)

In [ ]:
from collections import Counter

counts = Counter(encoding_data["raw_labels"])
df_counts = pd.DataFrame(counts.items(), columns=["Student", "Images Encoded"]).sort_values("Student")
display(df_counts)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df_counts["Student"], df_counts["Images Encoded"], color="#4C72B0")
ax.set_ylabel("Number of encoded images")
ax.set_title("Dataset composition per student")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

if df_counts["Images Encoded"].min() < 10:
    print("\n⚠ NOTE: Thesis-recommended minimum is 10 images/student. "
          f"Current minimum is {df_counts['Images Encoded'].min()} — "
          "see limitations discussion at the end of this notebook.")

## 3. Per-model metrics

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 Score (macro)"],
    "SVM": [svm_results["accuracy"], svm_results["precision"],
            svm_results["recall"], svm_results["f1"]],
    "KNN": [knn_results["accuracy"], knn_results["precision"],
            knn_results["recall"], knn_results["f1"]],
})
metrics_df[["SVM", "KNN"]] = metrics_df[["SVM", "KNN"]].round(4)
display(metrics_df)

## 4. Comparison chart — SVM vs KNN

In [ ]:
x = np.arange(len(metrics_df))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width/2, metrics_df["SVM"], width, label="SVM", color="#4C72B0")
ax.bar(x + width/2, metrics_df["KNN"], width, label="KNN", color="#DD8452")

ax.set_xticks(x)
ax.set_xticklabels(metrics_df["Metric"], rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("SVM vs KNN — Classifier Performance Comparison")
ax.legend()

for i, (svm_val, knn_val) in enumerate(zip(metrics_df["SVM"], metrics_df["KNN"])):
    ax.text(i - width/2, svm_val + 0.02, f"{svm_val:.2f}", ha="center", fontsize=9)
    ax.text(i + width/2, knn_val + 0.02, f"{knn_val:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(MODELS_DIR / "svm_vs_knn_comparison.png", bbox_inches="tight")
plt.show()
print(f"Saved chart to {MODELS_DIR / 'svm_vs_knn_comparison.png'}")

## 5. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (name, res) in zip(axes, [("SVM", svm_results), ("KNN", knn_results)]):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=res["confusion_matrix"],
        display_labels=class_names,
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} — Confusion Matrix")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(MODELS_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()
print(f"Saved chart to {MODELS_DIR / 'confusion_matrices.png'}")

## 6. Full classification reports (per-student precision/recall/F1)

In [ ]:
print("SVM Classification Report\n" + "=" * 50)
print(svm_results["classification_report"])

print("\nKNN Classification Report\n" + "=" * 50)
print(knn_results["classification_report"])

## 7. Discussion / Limitations (for thesis Results & Discussion chapter)

- **Dataset size**: The model comparison above is computed on a held-out
  test split drawn from the registered students' images. With few students
  and only 2-3 images each, the test set per class is extremely small
  (often a single image). A single misclassification can shift accuracy
  by 10-30+ percentage points — these results demonstrate the *evaluation
  methodology* (proper train/test split, multiple metrics, confusion
  matrices) rather than a statistically reliable accuracy claim.
- **Thesis-recommended minimum**: The project's own design recommends at
  least 10 images per registered student for robust evaluation; the actual
  deployment dataset falls below this, which should be explicitly stated
  as a limitation, with a recommendation for future work to expand the
  dataset before drawing strong conclusions about real-world accuracy.
- **SVM vs KNN choice for deployment**: Compare the metrics table above —
  whichever model scores higher on macro-F1 (which balances precision and
  recall across all students equally) is generally the safer choice to
  deploy in `app/recognizer.py`, but with this few test samples, treat the
  difference between the two models as indicative rather than conclusive.